# Работа с базами данных (2)

__Автор задач: Блохин Н.В. (NVBlokhin@fa.ru)__

Материалы:
* Макрушин С.В. Лекция "Работа с базами данных"
* https://sqliteonline.com/
* https://docs.python.org/3/library/sqlite3.html
* https://www.sqlitetutorial.net/sqlite-index/
* https://docs.python.org/3/library/sqlite3.html#sqlite3.IntegrityError
* https://www.sqlitetutorial.net/sqlite-alter-table/
* https://www.sqlitetutorial.net/sqlite-create-view/
* https://habr.com/ru/post/664000/
* https://learnsql.com/blog/what-is-common-table-expression/


## Задачи для совместного разбора

In [2]:
import pandas as pd
import sqlite3

# данные
students = pd.DataFrame(
    [
        ("Сотников Евгений Янович", 1),
        ("Степанова Виктория Константиновна", 1),
        ("Горелова Вероника Яновна", 2),
        ("Гришин Иван Романович", 3),
    ],
    columns=["name", "group_id"],
)
groups = list(zip([1, 2, 3], ["ПМ20-1", "ПМ20-2", "ПМ20-3"]))

con = sqlite3.connect("demo.sqlite")
con.execute("PRAGMA foreign_keys = 1")
cur = con.cursor()

# создаем таблицы
sql = """
DROP TABLE IF EXISTS StudentGroup;
DROP TABLE IF EXISTS Student;
CREATE TABLE StudentGroup (
    id int PRIMARY KEY,
    name varchar
);

CREATE TABLE Student(
    name VARCHAR PRIMARY KEY,
    group_id INT,
    FOREIGN KEY (group_id) REFERENCES StudentGroup(id)
);
"""
cur.executescript(sql)
con.commit()

# добавляем записи
sql = """
INSERT INTO StudentGroup(id, name) VALUES (?, ?)
"""
cur.executemany(sql, groups)
con.commit()

students.to_sql("Student", con, if_exists="append", index=False)

IntegrityError: FOREIGN KEY constraint failed

1\. Добавить столбец Age со значением по умолчанию. Добавить запись к таблицу

In [2]:
sql = '''
ALTER TABLE Student 
    ADD COLUMN Age INT DEFAULT 18
'''
cur.execute(sql)
con.commit()

In [3]:
pd.read_sql_query("SELECT * FROM Student", con)

,name,group_id,Age
0,Сотников Евгений Янович,1,18
1,Степанова Виктория Константиновна,1,18
2,Горелова Вероника Яновна,2,18
3,Гришин Иван Романович,3,18


In [4]:
sql = '''
INSERT INTO Student(name, group_id, age) 
    VALUES ("Баззаев Олег Давидович", 1, 30)
'''
try:
    cur.execute(sql)
except sqlite3.OperationalError as e:
    print("Произошла ошибка ", e)
else:
    print("Ошибки не было")
    con.commit()

Ошибки не было


2\. Занумеруйте студентов в рамках каждой группы.

In [5]:
sql = '''
SELECT name,
       group_id,
       age,
       ROW_NUMBER() OVER(PARTITION BY group_id ORDER BY name ASC) as rid
FROM Student
'''

In [6]:
pd.read_sql_query(sql, con)

,name,group_id,Age,rid
0,Баззаев Олег Давидович,1,30,1
1,Сотников Евгений Янович,1,18,2
2,Степанова Виктория Константиновна,1,18,3
3,Горелова Вероника Яновна,2,18,1
4,Гришин Иван Романович,3,18,1


3\. Выведите уникальные номера студентов

In [7]:
sql = '''
SELECT DISTINCT rid
FROM (
    SELECT name,
           group_id,
           age,
           ROW_NUMBER() OVER(PARTITION BY group_id ORDER BY name ASC) as rid
    FROM Student
)
'''

In [8]:
pd.read_sql_query(sql, con)

,rid
0,1
1,2
2,3


In [9]:
sql = '''
CREATE VIEW StudentView AS 
    SELECT name,
           group_id,
           age,
           ROW_NUMBER() OVER(PARTITION BY group_id ORDER BY name ASC) as rid
    FROM Student
'''
cur.execute(sql)
con.commit()

In [10]:
sql = '''
SELECT DISTINCT rid
FROM StudentView
'''
pd.read_sql_query(sql, con)

,rid
0,1
1,2
2,3


In [11]:
sql = '''
WITH StudentCTE AS (
    SELECT name,
           group_id,
           age,
           ROW_NUMBER() OVER(PARTITION BY group_id ORDER BY name ASC) as rid
    FROM Student
)

SELECT DISTINCT rid
FROM StudentCTE
'''
pd.read_sql_query(sql, con)

,rid
0,1
1,2
2,3


In [12]:
sql = '''
SELECT DISTINCT rid
FROM StudentCTE
'''
pd.read_sql_query(sql, con)

DatabaseError: Execution failed on sql '
SELECT DISTINCT rid
FROM StudentCTE
': no such table: StudentCTE

## Лабораторная работа 4

__При решении данных задач не подразумевается использования циклов или генераторов Python в ходе работы с пакетами `numpy` и `pandas`, если в задании не сказано обратного. Решения задач, в которых для обработки массивов `numpy` или структур `pandas` используются явные циклы (без согласования с преподавателем), могут быть признаны некорректными и не засчитаны.__

__Для начала работы подключитесь к БД `recipes.db` и создайте объект-курсор.__

In [3]:
con = sqlite3.connect(r"data\recipes.db")
cur = con.cursor()

<p class="task" id="1"></p>

1\. Создайте уникальный индекс для таблицы `Review` для обеспечения уникальности сочетания значений в полях `user_id` и `recipe_id`. 

In [4]:
sql = '''
CREATE UNIQUE INDEX idx_user_recipe
ON Review(user_id, recipe_id);
'''

cur.execute(sql)
con.commit()

In [8]:
sql = '''
PRAGMA index_list('Review');
'''
pd.read_sql_query(sql, con)

,seq,name,unique,origin,partial
0,0,idx_user_recipe,1,c,0
1,1,sqlite_autoindex_Review_1,1,pk,0


<p class="task" id="2"></p>

2\. Напишите функцию `add_review(review_id, user_id, recipe_id, date, rating, review)`, которая добавляет запись в таблицу `Review`. В случае успешного добавления функция должна вернуть значение 0. В случае нарушения ограничения целостности функция должна вернуть значение 1. В случае любых других ошибок функция должна вернуть значение 2. Продемонстрируйте работу функции, попытавшись добавить одну и ту же запись дважды в двух ячейках подряд.

Для решения задачи воспользуйтесь механизмом try - except и обработайте соответствующее исключение.

In [7]:
def add_review(review_id, user_id, recipe_id, date, rating, review):
    sql = '''
    INSERT INTO Review
    VALUES (?, ?, ?, ?, ?, ?);
    '''
    try:
        cur.execute(sql, (review_id, user_id, recipe_id, date, rating, review))
        con.commit()
        return 0
    except sqlite3.IntegrityError:
        return 1
    except:
        return 2

In [8]:
add_review(15103213, 1333, 1567, '2009-10-10', 4, 'nice!')

0

In [9]:
add_review(15103213, 1333, 1567, '2009-10-10', 4, 'nice!')

1

<p class="task" id="3"></p>

3\. _Измените_ таблицу Review, добавив в нее поле `toxic` булева типа. 

In [10]:
sql = '''
ALTER TABLE Review
ADD COLUMN toxic BOOLEAN;
'''
cur.execute(sql)
con.commit()

In [11]:
pd.read_sql_query("SELECT * FROM Review;", con)

,id,user_id,recipe_id,date,rating,review,toxic
0,370476,21752,57993,2003-05-01,5,Last week whole sides of frozen salmon fillet ...,None
1,624300,431813,142201,2007-09-16,5,So simple and so tasty! I used a yellow capsi...,None
2,187037,400708,252013,2008-01-10,4,"Very nice breakfast HH, easy to make and yummy...",None
3,706134,2001852463,404716,2017-12-11,5,These are a favorite for the holidays and so e...,None
4,312179,95810,129396,2008-03-14,5,Excellent soup! The tomato flavor is just gre...,None
...,...,...,...,...,...,...,...
126692,158736,2282344,8701,2012-06-03,0,This recipe is outstanding. I followed the rec...,None
126693,1059834,689540,222001,2008-04-08,5,"Well, we were not a crowd but it was a fabulou...",None
126694,453285,2000242659,354979,2015-06-02,5,I have been a steak eater and dedicated BBQ gr...,None
126695,691207,463435,415599,2010-09-30,5,Wonderful and simple to prepare seasoning blen...,None


<p class="task" id="4"></p>

4\. Вам дан классификатор `clf`, который классифицирует тексты отзывов как токсичные (`True`) и не токсичные (`False`).
Напишите функцию `classify_reviews`, которая итеративно получает пакет (батч) `batch_size` строк из таблицы Reviews, у которых не проставлено значение в столбце `toxic`, делает для них прогноз при помощи модели `clf` и обновляет соответствующие строки в БД. Данная процедура выполняется до тех пор, пока в БД есть строки, для которых требуется получить прогноз.

Продемонстрируйте результат, выведя на экран количество токсичных и не токсичных отзывов в таблице.

In [12]:
from sklearn.dummy import DummyClassifier

clf = DummyClassifier(strategy="uniform").fit(None, [True, False])

In [13]:
def classify_reviews(batch_size, clf):   
    while True:
        sql = f'''
        SELECT * 
        FROM Review
        WHERE toxic IS NULL
        LIMIT {batch_size};
        '''
        batch_df = pd.read_sql(sql, con)
        if batch_df.empty:
            break
        predictions = clf.predict(batch_df['review'])
        for i, pred in enumerate(predictions):
            review_id = batch_df.iloc[i]['id']
            sql = f'''
            UPDATE Review 
            SET toxic = {pred} 
            WHERE id = {review_id};
            '''
            cur.execute(sql)
        con.commit()

In [ ]:
def classify_reviews():
    batch = pd.read_sql_query(
        'SELECT * FROM Review WHERE toxic IS NULL LIMIT ?', 
        con,
        params=(10_000, )
    )
    
    predictions = clf.predict(batch)
    batch_ids = batch_size.id
    
#     for i in range(batch_size.shape[0]):
    while not batch.empty:
        prediction = predictions
        batch_id = batch_ids["id"]
        sql = '''
                 UPDATE Review
                 SET toxic = ?
                 WHERE
                     id == ?;
                 '''

        args = zip(prediction, batch_id)
        cur.executemany(sql, args)
        con.commit()
        batch = pd.read_sql_query(
            'SELECT * FROM Review WHERE toxic IS NULL LIMIT ?', 
            con,
            params=(10_000, )
        )

        predictions = clf.predict(batch)
        batch_ids = batch_size.id
        
    return pd.read_sql_query("SELECT * FROM Review", con)

In [14]:
classify_reviews(1000, clf)

In [15]:
pd.read_sql_query("SELECT * FROM Review;", con)['toxic'].value_counts()

1    63387
0    63310
Name: toxic, dtype: int64

<p class="task" id="5"></p>

5\. Создайте представление `RecipeWithYear`, в котором добавлен дополнительный столбец `year`, содержащий год даты из столбца `submitted`. Сделайте выборку из этого представления и выведите на экран количество рецептов с разбивкой по годам.

In [16]:
pd.read_sql_query("SELECT * FROM Recipe;", con)

,id,name,minutes,submitted,description,n_ingredients
0,44123,george s at the cove black bean soup,90,2002-10-25,an original recipe created by chef scott meska...,18.0
1,67664,healthy for them yogurt popsicles,10,2003-07-26,my children and their friends ask for my homem...,NaN
2,38798,i can t believe it s spinach,30,2002-08-29,"these were so go, it surprised even me.",8.0
3,35173,italian gut busters,45,2002-07-27,my sister-in-law made these for us at a family...,NaN
4,84797,love is in the air beef fondue sauces,25,2004-02-23,i think a fondue is a very romantic casual din...,NaN
...,...,...,...,...,...,...
29995,267661,zurie s holey rustic olive and cheddar bread,80,2007-11-25,this is based on a french recipe but i changed...,10.0
29996,386977,zwetschgenkuchen bavarian plum cake,240,2009-08-24,"this is a traditional fresh plum cake, thought...",11.0
29997,103312,zwiebelkuchen southwest german onion cake,75,2004-11-03,this is a traditional late summer early fall s...,NaN
29998,486161,zydeco soup,60,2012-08-29,this is a delicious soup that i originally fou...,NaN


In [26]:
sql = '''
CREATE VIEW RecipeWithYear AS
SELECT *, CAST(SUBSTR(submitted, 1, 4) AS INT) AS year
FROM Recipe;
'''

In [27]:
cur.execute(sql)
con.commit()

In [29]:
sql = '''
SELECT year, COUNT(*) AS count
FROM RecipeWithYear
GROUP BY year;
'''

pd.read_sql_query(sql, con)

,year,count
0,1999,275
1,2000,104
2,2001,589
3,2002,2644
4,2003,2334
5,2004,2153
6,2005,3130
7,2006,3473
8,2007,4429
9,2008,4029


<p class="task" id="6"></p>

6\. Напишите запрос на языке SQL, который возвращает все строки из таблицы `Recipe` с дополнительным столбцом, содержащем номер рецепта. Рецепты нумеруются целыми числами, начиная с 1, в __рамках каждого года__ в порядке их добавления в БД (столбец `submitted`). Получите результат в виде `pd.DataFrame`. Посчитайте и выведите на экран количество строк полученного `pd.DataFrame`, для которых сгенерированный номер кратен 50.

In [34]:
sql = '''
SELECT *, ROW_NUMBER() OVER(PARTITION BY CAST(SUBSTR(submitted, 1, 4) AS INT) ORDER BY submitted) as recipe_number
FROM Recipe;
'''

In [36]:
recipes_with_numbers = pd.read_sql_query(sql, con)
recipes_with_numbers

,id,name,minutes,submitted,description,n_ingredients,recipe_number
0,203,chinese plum sauce,115,1999-08-06,chinese plum sauce serve this with egg rolls ...,12.0,1
1,653,b c cherry and raspberry preserves,215,1999-08-09,None,4.0,2
2,360,baked zucchini frittatas,67,1999-08-09,None,NaN,3
3,658,dried fruit roll ups,1495,1999-08-09,fruit roll-ups,4.0,4
4,1144,steak tomato basil pasta,0,1999-08-09,None,11.0,5
...,...,...,...,...,...,...,...
29995,536547,cauliflower ceviche,45,2018-07-30,a healthy ceviche - a perfect appetizer for pa...,8.0,20
29996,536610,miracle home made puff pastry,35,2018-07-31,puff pastry that you can make in minutes? at h...,NaN,21
29997,536729,creole watermelon feta salad,10,2018-08-11,spicy watermelon salad. from tony chachere's s...,NaN,22
29998,536728,gluten free vegemite,2,2018-08-11,gluten free vegemite-like stuff.,3.0,23


In [41]:
len(recipes_with_numbers[recipes_with_numbers['recipe_number'] % 50 == 0])

589

<p class="task" id="7"></p>

7\. Используя обобщенное табличное выражение и решение задачи 6, напишите запрос на языке SQL, который вернет количество строк, для которых сгенерированный номер кратен 50. Выполните запрос и выведите количество таких строк на экран.

In [52]:
sql = '''
WITH  Recipes_with_numbers AS (
    SELECT *, ROW_NUMBER() OVER(PARTITION BY CAST(SUBSTR(submitted, 1, 4) AS INT)) AS recipe_number
    FROM Recipe
)
SELECT COUNT(*) AS count
FROM Recipes_with_numbers
WHERE recipe_number % 50 = 0
'''

pd.read_sql_query(sql,con)

,count
0,589


In [54]:
cur.close()
con.close()